# 🏗️ Short Design Report — Multi-Tenant Project Management SaaS trên Microsoft Azure

**Đề tài 06 (Advanced) · Môn Điện toán đám mây · Nhóm 4 thành viên**
Stack: Azure · Java 21 / Spring Boot 3 · PostgreSQL 16 · React (Vite + TS) · Terraform azurerm · GitHub Actions

| Người tổng hợp | Người cung cấp nội dung |
|---|---|
| TV4 | TV1 (hạ tầng, chi phí, CI/CD) · TV2 (API, DB, RLS) · TV3 (Worker, giám sát, k6) |

> **Quy ước:** chỗ ghi `⟦ĐIỀN: …⟧` là **số liệu đo thật**, do thành viên phụ trách cung cấp sau đợt chạy Azure (Tuần 6).
> Không điền số ước đoán vào đó.

### Mục lục
1. Bài toán · 2. Kiến trúc & ranh giới mạng · 3. Thiết kế Multi-Tenant · 4. Kết quả kiểm chứng bảo mật ·
5. Deployment & Automation · 6. Reliability & Observability · 7. Câu hỏi phản biện · 8. Chi phí & Cleanup · 9. Giới hạn

---
## 1. Bài toán & tiêu chí thành công

Website quản lý dự án/công việc (kiểu Trello/Jira thu nhỏ) phục vụ **nhiều tổ chức (tenant) cùng lúc**. Mỗi tenant có
thành viên, dự án, task board, audit log và export CSV riêng.

> 🔴 **Yêu cầu sống còn:** không rò rỉ dữ liệu giữa các tenant ở **bất kỳ tầng nào** (API, DB, cache, blob, worker, log),
> kể cả khi người dùng cố tình sửa ID trên API hoặc giả mạo job trong hàng đợi.

| Vai trò | Quyền |
|---|---|
| **Owner** | Toàn quyền org: mời/xóa thành viên, đổi vai trò, CRUD dự án/task, export |
| **Member** | CRUD dự án/task, xem lịch sử, export |
| **Guest** | Chỉ đọc. Mọi request ghi bị **API** trả `403` |

---
## 2. Kiến trúc

```text
 Trình duyệt ──HTTPS──► Azure Static Web Apps (React SPA)
     │  OIDC + PKCE
     ├────────────────► Microsoft Entra External ID (CIAM) ──► JWT access token
     │  Bearer JWT, X-Request-Id
     ▼
 App Service (Linux B1, Java 21) ── api profile ──┐          ┌── worker profile (cùng Plan B1)
     │ VNet Integration (snet-app 10.0.1.0/24)    │          │
     ├──► PostgreSQL Flexible B1ms (snet-db 10.0.2.0/24, private DNS, public access = OFF)
     ├──► Storage Account: Queue `export-jobs` (+ `export-jobs-poison`) ──► Worker
     │                     Blob private `exports/{tenant_id}/{job_id}.csv` ◄── Worker
     ├──► Key Vault (RBAC) ◄── Managed Identity (Key Vault reference cho DB password)
     └──► Application Insights / Log Analytics ──► Dashboard 4 biểu đồ, Metric Alert ──► Action Group (Email)
```

⟦ĐIỀN: chèn **Actual Architecture Diagram** của TV1 (ảnh từ Azure Portal / draw.io sau `terraform apply`)⟧

### 2.1 Ranh giới mạng & bảo vệ bí mật

| Thành phần | Biện pháp | Ở đâu trong code |
|---|---|---|
| 🗄️ PostgreSQL | `public_network_access_enabled = false`, subnet delegate, Private DNS Zone. Chỉ App Service (qua VNet Integration) kết nối được | `infra/postgresql.tf`, `infra/network.tf` |
| 📦 Blob | `allow_nested_items_to_be_public = false`, container `private`, TLS ≥ 1.2. Tải qua **User Delegation SAS** read-only, ≤ 5 phút, scope 1 blob | `infra/storage_queue.tf` |
| 🔑 Secrets | Managed Identity; DB password chỉ có trong Key Vault (`@Microsoft.KeyVault(SecretUri=…)`); Storage dùng RBAC, không account key; SPA là public client (PKCE), không client secret | `infra/keyvault.tf`, `infra/app_service.tf` |
| 🔒 HTTPS | `https_only = true`; SWA mặc định HTTPS; CSP, HSTS, `X-Frame-Options: DENY`, `Referrer-Policy: no-referrer` (để SAS không lọt qua Referer) | `frontend/public/staticwebapp.config.json` |

Cell dưới đây **đọc trực tiếp Terraform** để kiểm tra các khẳng định trên còn đúng với code hiện tại:

In [1]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists() or (p / "infra").is_dir())
print("Repo root:", ROOT.name)

Repo root: multi-tenant-project-management


In [2]:
import re
infra = {p.name: p.read_text(encoding="utf-8") for p in (ROOT / "infra").glob("*.tf")}
checks = [
    ("DB tắt public access",         "postgresql.tf",    r"public_network_access_enabled\s*=\s*false"),
    ("Blob cấm anonymous",           "storage_queue.tf", r"allow_nested_items_to_be_public\s*=\s*false"),
    ("Container exports private",    "storage_queue.tf", r'container_access_type\s*=\s*"private"'),
    ("TLS tối thiểu 1.2",            "storage_queue.tf", r'min_tls_version\s*=\s*"TLS1_2"'),
    ("App Service HTTPS only",       "app_service.tf",   r"https_only\s*=\s*true"),
    ("DB password qua Key Vault ref","app_service.tf",   r"@Microsoft\.KeyVault"),
    ("Managed Identity",             "app_service.tf",   r'type\s*=\s*"SystemAssigned"'),
    ("Storage tắt public network",   "storage_queue.tf", r"public_network_access_enabled\s*=\s*false"),
    ("Có Static Web App",            "*",                r'resource\s+"azurerm_static_web_app"'),
]
for label, f, rx in checks:
    text = "\n".join(infra.values()) if f == "*" else infra.get(f, "")
    print(("✅" if re.search(rx, text) else "❌ CHƯA CÓ"), label)

✅ DB tắt public access
✅ Blob cấm anonymous
✅ Container exports private
✅ TLS tối thiểu 1.2
✅ App Service HTTPS only
✅ DB password qua Key Vault ref
✅ Managed Identity
❌ CHƯA CÓ Storage tắt public network
❌ CHƯA CÓ Có Static Web App


---
## 3. Thiết kế Multi-Tenant (phòng thủ nhiều lớp)

Mô hình **shared database, shared schema**: mọi bảng nghiệp vụ có `tenant_id`. Đây là lựa chọn rẻ nhất cho B1ms, đổi lại
isolation phải được bảo đảm bằng **nhiều lớp độc lập**, để một lớp có lỗi thì lớp sau vẫn chặn được.

| # | Lớp | Cơ chế | Chặn được |
|---|---|---|---|
| 1 | Xác thực | Spring Security verify JWT Entra (issuer, audience, chữ ký, hạn) | Không token / token giả `alg=none` → `401` |
| 2 | Ủy quyền theo tenant | `TenantMembershipInterceptor`: đọc `tenantId` từ path, tra **memberships mỗi request**, kiểm role | Org không thuộc về mình → `403`; Guest ghi → `403`; thu hồi có hiệu lực **ngay** |
| 3 | Truy vấn có phạm vi | `WHERE tenant_id = :verifiedTenant AND id = :id` | IDOR đổi ID → `404` |
| 4 | PostgreSQL RLS | `SET LOCAL app.tenant_id` + policy `tenant_id = current_setting('app.tenant_id')::uuid` | Code quên điều kiện tenant vẫn bị DB chặn |
| 5 | Ràng buộc dữ liệu | `UNIQUE(tenant_id, name)` trên `projects` | Hai tenant trùng tên không xung đột (Test 3) |
| 6 | Cache | Khóa `tenant:{id}:…`, invalidate theo tenant | Lẫn dữ liệu qua cache (Test 3) |
| 7 | Blob | Path `exports/{tenant_id}/{job_id}.csv` lấy từ DB, SAS scope 1 blob | Đoán/tải file tenant khác |
| 8 | Worker | Message queue **không đáng tin**: đối chiếu `claimedTenantId` với `export_jobs.tenant_id`, kiểm lại membership | Job giả mạo → từ chối, log `jobId`, metric, Alert |
| 9 | Log | JSON log + MDC `requestId/jobId/tenantId`; redact JWT/password/SAS | Rò rỉ bí mật qua log |

⟦ĐIỀN (TV2): SQL policy RLS thật, tên role DB của API/Worker, cách reset tenant context trên HikariCP⟧

### 3.1 Mô hình dữ liệu (6 bảng)

| Bảng | Cột chính | Ghi chú |
|---|---|---|
| `tenants` | `id, name` | |
| `memberships` | `tenant_id, user_id, role` | Nguồn sự thật cho quyền, được kiểm mỗi request |
| `projects` | `tenant_id, name, …` | `UNIQUE(tenant_id, name)` |
| `tasks` | `tenant_id, project_id, title, status, assignee` | |
| `audit_events` | `tenant_id, actor, action, entity, request_id, details JSONB` | |
| `export_jobs` | `tenant_id, requested_by, status, blob_path` | **Nguồn sự thật** cho Worker |

⟦ĐIỀN (TV2): ERD + migration Flyway⟧

### 3.2 Luồng Export bất đồng bộ
1. `POST /orgs/{t}/exports` → API ghi `export_jobs(QUEUED)` + audit → gửi message `{schemaVersion, jobId, claimedTenantId, requestId}` → `202 {jobId}`.
2. Worker nhận message, tra `export_jobs` theo `jobId`, **so tenant**, kiểm membership, claim job nguyên tử.
3. Query tasks bằng tenant **đã xác minh từ DB** → CSV (chống formula injection) → Blob private → `COMPLETED`.
4. UI poll `GET /exports/{jobId}`; xong thì `GET …/download` → SAS 5 phút → trình duyệt tải thẳng từ Blob.

---
## 4. Kết quả kiểm chứng bảo mật

Bộ test tự động `tests/security/` (pytest, gọi thẳng API, ghi raw HTTP + timestamp, tự che token).
Bộ test đã được **tự kiểm chứng** bằng mock API cố tình có lỗ hổng: mock an toàn cho 34/34 PASS, còn mỗi lỗ hổng
(IDOR, membership cũ, unique toàn cục, cache dùng chung) đều làm test tương ứng FAIL. Xem notebook `STATUS_AND_HANDOFF.ipynb`.

| Test | Kịch bản | Kết quả trên Azure | Bằng chứng |
|---|---|---|---|
| Test 1: IDOR | 13 kiểu đọc + 11 kiểu ghi/xóa/export bằng JWT Org A lên ID Org B; token giả; Guest ghi | ⟦ĐIỀN⟧ | `docs/tv4/evidence/<run>/test_org_a_cannot_*.http.txt` |
| Test 2: Thu hồi membership | Owner xóa X → dùng lại JWT cũ của X | ⟦ĐIỀN: status = 403, khoảng cách ms⟧ | `…/test_revoked_member_is_blocked_on_next_request.http.txt` |
| Test 3: Trùng tên & cache | A và B cùng tạo "Cloud Project", đọc 2 lần, sửa/xóa ở A | ⟦ĐIỀN⟧ | `…/test_same_project_name_in_two_tenants.http.txt` |
| Failure Drill (TV3) | Message có `claimedTenantId` của org khác | ⟦ĐIỀN: log jobId, metric, email Alert lúc hh:mm, API vẫn 200⟧ | ⟦ĐIỀN: ảnh⟧ |

Cell dưới **tự đọc** bằng chứng đã rà soát trong `docs/tv4/evidence/` (sau Đợt 2 sẽ tự hiện kết quả Azure):

In [3]:
import json
ev_root = ROOT / "docs" / "tv4" / "evidence"
runs = sorted(ev_root.glob("*/summary.json")) if ev_root.exists() else []
if not runs:
    print("⏳ Chưa có bằng chứng Azure trong docs/tv4/evidence/ (sẽ có sau Đợt 2, Tuần 6).")
for s in runs:
    data = json.loads(s.read_text(encoding="utf-8"))
    info = json.loads((s.parent / "run-info.json").read_text(encoding="utf-8"))
    print(f"📁 {s.parent.name}  target={info['target']}  PASS {data['passed']}/{data['passed'] + data['failed']}")
    for group in ("test_1", "test_2", "test_3"):
        rs = [r for r in data["results"] if r["test"].startswith(group)]
        if rs:
            ok = sum(r["outcome"] == "passed" for r in rs)
            print(f"   {group}: {ok}/{len(rs)} PASS")

⏳ Chưa có bằng chứng Azure trong docs/tv4/evidence/ (sẽ có sau Đợt 2, Tuần 6).


---
## 5. Deployment & Automation

| Hạng mục | Cách làm | Trạng thái |
|---|---|---|
| IaC | Toàn bộ hạ tầng trong `infra/` (Terraform azurerm ~> 4.0); `apply` dựng từ đầu, `destroy` xóa sạch | Đã có (TV1) |
| CI/CD | GitHub Actions + **OIDC Federated Credentials** (không lưu client secret trong GitHub) | ⟦ĐIỀN (TV1)⟧ |
| Rollback | Deployment Slot swap hoặc redeploy artifact/tag trước | ⟦ĐIỀN (TV1): thời gian rollback, link video⟧ |

---
## 6. Reliability & Observability

- Log JSON có `request_id` (API) và `job_id` (Worker). Mỗi lỗi trên UI hiển thị `request_id` để tra ngược.
- **Dashboard 4 biểu đồ:** ⟦ĐIỀN (TV3): ví dụ request rate & 5xx, p95 latency, độ sâu queue/poison, số tenant-mismatch⟧
- **Alert:** Metric Alert → Action Group Email.
  ⚠️ Terraform hiện chỉ có alert `Http5xx` của API. Failure Drill cần thêm alert trên **metric tenant-mismatch** của Worker (TV3/TV1).
- **Load test k6** (2–3 phút, có warmup): ⟦ĐIỀN (TV3): p95 = … ms, error rate = … %, số VU⟧

Truy vết một request từ file bằng chứng sang App Insights:
```kusto
union traces, requests
| where customDimensions.request_id == "tv4-A-owner-..."
| order by timestamp asc
```

---
## 7. Câu hỏi phản biện

### 7.1 Scaling: chuyện gì xảy ra khi tải tăng 10×?

| Thành phần | Hành vi khi scale | Hướng xử lý |
|---|---|---|
| API | **Stateless** (JWT + membership trong DB) nên scale ngang ngay | Cache in-process → Azure Cache for Redis (giữ khóa `tenant:{id}:…`) khi nhiều instance |
| Worker | Scale độc lập; Queue hấp thụ đột biến, API không bị chậm | Tách Plan / Container Apps scale theo độ dài queue |
| **PostgreSQL B1ms** | **Nút cổ chai đầu tiên** (1 vCPU burstable, ít kết nối) | Index `(tenant_id, …)`, phân trang → General Purpose → read replica → PgBouncer |
| Noisy neighbor | 1 tenant export lớn làm nghẽn tenant khác | Giới hạn job đồng thời/tenant, rate limit theo tenant, stream CSV |
| Tenant doanh nghiệp | Cần tách biệt vật lý | DB riêng (silo), lớp 2–3 giữ nguyên, thêm định tuyến connection |

### 7.2 Connection Pool

**Ràng buộc:** `số instance × (pool API + pool Worker) + dự trữ hệ thống ≤ max_connections`

- Pool nhỏ + transaction ngắn tốt hơn pool lớn: 1 vCPU không chạy song song nhiều truy vấn hơn số core.
- ⚠️ **Rủi ro riêng của multi-tenant:** connection dùng lại có thể mang `app.tenant_id` của request trước.
  Biện pháp: `SET LOCAL` (chỉ sống trong transaction), hoặc reset khi trả về pool.
  Test 1 và 3 xen kẽ hai tenant trên cùng pool để phát hiện lỗi này.
- Scale-out nhân số kết nối lên → giảm pool/instance hoặc bật PgBouncer.

⟦ĐIỀN (TV2): `maximumPoolSize` thực tế và kết quả `SHOW max_connections;` trên server⟧

▶️ Đổi tham số trong cell dưới để xem cấu hình nào an toàn khi scale:

In [4]:
# Tham số — TV2 cập nhật bằng số thật (max_connections lấy từ: SHOW max_connections;)
MAX_CONNECTIONS = 50      # giá trị mặc định tham khảo cho Burstable B1ms — cần xác nhận trên server thật
RESERVED        = 5       # dự trữ cho admin/hệ thống Azure, migration, psql khi debug
API_POOL        = 10      # spring.datasource.hikari.maximum-pool-size của profile api
WORKER_POOL     = 3       # của profile worker

print(f"{'API inst':>8} {'Worker inst':>11} {'Tổng kết nối':>13} {'Giới hạn':>9}  Kết luận")
for api_n, worker_n in [(1, 1), (2, 1), (3, 1), (4, 2), (6, 2)]:
    total = api_n * API_POOL + worker_n * WORKER_POOL
    limit = MAX_CONNECTIONS - RESERVED
    verdict = "✅ an toàn" if total <= limit else f"❌ vượt {total - limit} → giảm pool hoặc dùng PgBouncer"
    print(f"{api_n:>8} {worker_n:>11} {total:>13} {limit:>9}  {verdict}")

API inst Worker inst  Tổng kết nối  Giới hạn  Kết luận
       1           1            13        45  ✅ an toàn
       2           1            23        45  ✅ an toàn
       3           1            33        45  ✅ an toàn
       4           2            46        45  ❌ vượt 1 → giảm pool hoặc dùng PgBouncer
       6           2            66        45  ❌ vượt 21 → giảm pool hoặc dùng PgBouncer


### 7.3 Retry Safety / Idempotency

Azure Queue giao hàng **at-least-once**: message có thể đến 2 lần, hoặc đến sau khi Worker chết giữa chừng.

| Tình huống | Vì sao vẫn an toàn |
|---|---|
| Message đến 2 lần | Claim nguyên tử: `UPDATE export_jobs … WHERE status IN ('QUEUED', lease hết hạn)`; job đã `COMPLETED` thì bỏ qua |
| Worker chết khi đang ghi blob | Path blob xác định theo `jobId`: ghi lại thì ghi đè đúng file, không sinh rác |
| Worker chết trước khi commit | Chỉ xóa message **sau** khi commit kết quả cuối → message hiện lại sau visibility timeout |
| Lỗi tạm thời (mạng, throttling) | Backoff + jitter; quá số lần thì vào `export-jobs-poison` với metadata đã làm sạch |
| Người dùng bấm Export 2 lần | Tạo 2 job độc lập, vô hại; có thể thêm header `Idempotency-Key` |

Chi tiết: `prototypes/tv3-worker/docs/CONTRACT_TO_CONFIRM.md`. ⟦ĐIỀN (TV3): kết quả test retry/poison⟧

### 7.4 Alternative Compute

| Lựa chọn | Ưu | Nhược | Kết luận |
|---|---|---|---|
| **App Service B1 (đã chọn)** | HTTPS sẵn, VNet Integration, Deployment Slot, Java 21, chi phí cố định thấp, API + Worker chung 1 Plan | Không scale về 0 | ✅ Phù hợp nhất với $100 và chiến lược bật theo đợt |
| Azure Container Apps | Scale về 0, scale Worker theo queue (KEDA), revision để rollback | Cold start Spring Boot, VNet phức tạp hơn | Ứng viên tốt nhất cho Worker khi tải thật |
| Azure Functions (Queue trigger) | Trả theo lần chạy, tự scale theo queue | Spring Boot nặng, giới hạn thời gian chạy, VNet cần Premium/Flex | Hợp với Worker nhỏ viết lại |
| AKS | Linh hoạt tối đa | Chi phí & vận hành vượt quy mô đồ án | ❌ Không chọn |

---
## 8. Chi phí & Cleanup

Region Southeast Asia. ⟦ĐIỀN (TV1): bảng Cost Estimate từ Azure Pricing Calculator⟧

| Tài nguyên | SKU | Bật 24/7 (USD/tháng) | Theo 3 đợt |
|---|---|---|---|
| App Service Plan | B1 Linux | ⟦ĐIỀN⟧ (~13 theo tài liệu nhóm) | ⟦ĐIỀN⟧ |
| PostgreSQL Flexible | B1ms + 32 GB | ⟦ĐIỀN⟧ | ⟦ĐIỀN⟧ |
| Storage | Standard LRS | ⟦ĐIỀN⟧ | ⟦ĐIỀN⟧ |
| Key Vault, App Insights, Log Analytics | Standard / PerGB | ⟦ĐIỀN⟧ | ⟦ĐIỀN⟧ |
| Static Web Apps, Entra External ID | Free (≤ 50.000 MAU) | 0 | 0 |
| **Tổng** | | ⟦ĐIỀN⟧ | ⟦ĐIỀN: thực tế từ Cost Management⟧ |

**Kiểm soát:** Cost Alert $25 / $50 / $75. 85% thời gian dev chạy local (Docker PostgreSQL + Azurite, hoặc mock) với chi phí $0.

**Cleanup checklist**
- [ ] `terraform destroy`
- [ ] Kiểm tra Resource Group trống trên Portal (chụp ảnh)
- [ ] Xóa app registration test nếu không dùng nữa
- [ ] Key Vault soft-delete 7 ngày: purge nếu cần tạo lại cùng tên

---
## 9. Giới hạn & hướng phát triển
- Cache in-process chưa dùng chung giữa nhiều instance (xem 7.1).
- Mời thành viên đang thêm trực tiếp theo email; production cần lời mời có xác nhận và hết hạn.
- Chưa có WAF/Front Door hay rate limit theo tenant.
- ⟦ĐIỀN: giới hạn thực tế phát hiện khi chạy Azure⟧